# Private MCP Server E2E Lab — Cloud Shell Notebook

This notebook tests Azure AI Foundry agents with private resources deployed by **Template 19** (Hybrid Private Resources Agent Setup).

## What This Notebook Tests

| # | Test | What It Validates |
|---|------|--------------------|
| 1 | OpenAI Responses API | Direct API connectivity to AI Services |
| 2 | Basic Agent | Agent creation + Responses API (no tools) |
| 3 | MCP Tool via Agent | Agent → Data Proxy → private MCP server (`get_weather`) |
| 4 | AI Search Tool via Agent | Agent → Data Proxy → private AI Search endpoint |
| 5 | Direct MCP Connectivity | Raw HTTP session flow to MCP server (VNet mode only) |

## Two Connectivity Modes

| Mode | How It Works | When to Use |
|------|-------------|-------------|
| **`public`** (default) | Temporarily enables public network access on AI Services so Cloud Shell can reach the API. Backend resources (AI Search, Cosmos DB, Storage, MCP server) remain private — the Data Proxy routes agent tool calls through the VNet. | Standard Cloud Shell without VNet integration |
| **`vnet`** | Cloud Shell is deployed into the same VNet (or peered). All resources are reachable directly. Enables the direct MCP HTTP connectivity test. | Cloud Shell with VNet integration configured |

## Architecture

```
Cloud Shell (you are here)
  │
  ├─[public mode]─► AI Services (public endpoint enabled)
  │                    └─► Data Proxy (networkInjections)
  │                          └─► Private VNet
  │                                ├─ mcp-subnet → weather-mcp Container App
  │                                └─ pe-subnet  → AI Search, Cosmos DB, Storage
  │
  └─[vnet mode]───► AI Services (private endpoint)
                     └─► Data Proxy → Private VNet (same as above)
```

## Prerequisites

- Template 19 deployed to a resource group (e.g., `rg-hybrid-agent-test`)
- MCP server deployed via `deploy-mcp.sh`
- Azure CLI authenticated (`az login`)
- Owner or Contributor role on the subscription

---
## Cell 1 — Configuration

Set your resource group name and connectivity mode. Everything else is auto-discovered.

In [ ]:
# ============================================================================
# USER CONFIGURATION — Edit these values
# ============================================================================

RESOURCE_GROUP = "rg-hybrid-agent-test"  # Your resource group name

CONNECTIVITY_MODE = "public"  # "public" = temporarily enable public access on AI Services
                               # "vnet"   = Cloud Shell has VNet integration (everything private)

MODEL_NAME = "gpt-4o-mini"  # Deployed model name

AI_SEARCH_INDEX_NAME = "test-index"  # AI Search index to query (created in Step 3 of TESTING-GUIDE)

MAX_RETRIES = 3  # Retry attempts for MCP agent test (Hyena routing workaround)

# ============================================================================
# Validation
# ============================================================================
assert CONNECTIVITY_MODE in ("public", "vnet"), f"Invalid CONNECTIVITY_MODE: {CONNECTIVITY_MODE!r}. Must be 'public' or 'vnet'."
assert RESOURCE_GROUP, "RESOURCE_GROUP must not be empty."
assert MODEL_NAME, "MODEL_NAME must not be empty."

print(f"Resource Group:    {RESOURCE_GROUP}")
print(f"Connectivity Mode: {CONNECTIVITY_MODE}")
print(f"Model:             {MODEL_NAME}")
print(f"Search Index:      {AI_SEARCH_INDEX_NAME}")
print(f"Max Retries:       {MAX_RETRIES}")
print("\n✓ Configuration OK")

---
## Cell 2 — Resource Discovery

Auto-discovers all Azure resource names and endpoints from the resource group using `az` CLI. Each value is validated — if anything is missing, you'll see exactly what failed.

In [ ]:
import subprocess, sys, json

def az_query(cmd: str, label: str) -> str:
    """Run an az CLI command and return stripped stdout. Raises on empty result."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    value = result.stdout.strip().strip('"')
    if not value:
        stderr_msg = result.stderr.strip()[:300] if result.stderr else "(no stderr)"
        print(f"\n✗ FAILED to discover: {label}")
        print(f"  Command: {cmd}")
        print(f"  Stderr:  {stderr_msg}")
        raise SystemExit(f"Resource discovery failed for '{label}'. Fix the issue above and re-run this cell.")
    return value

# --- Verify resource group exists ---
rg_location = az_query(
    f'az group show --name {RESOURCE_GROUP} --query location -o tsv',
    "Resource Group location"
)
print(f"✓ Resource Group '{RESOURCE_GROUP}' found in '{rg_location}'")

# --- AI Services account ---
AI_SERVICES_NAME = az_query(
    f'az cognitiveservices account list -g {RESOURCE_GROUP} --query "[0].name" -o tsv',
    "AI Services account name (is Template 19 deployed?)"
)
print(f"✓ AI Services: {AI_SERVICES_NAME}")

AI_SERVICES_ENDPOINT = az_query(
    f'az cognitiveservices account show -g {RESOURCE_GROUP} -n {AI_SERVICES_NAME} --query "properties.endpoint" -o tsv',
    "AI Services endpoint"
)
print(f"✓ AI Services Endpoint: {AI_SERVICES_ENDPOINT}")

# --- Project name (from AI Services sub-accounts / projects) ---
PROJECT_NAME = az_query(
    f'az cognitiveservices account list -g {RESOURCE_GROUP} --query "[?kind==\'AIServices\'].name | [0]" -o tsv',
    "AI Services account (kind=AIServices)"
)

# Discover the project under the AI Services account
_projects_json = subprocess.run(
    f'az rest --method get --url "https://management.azure.com/subscriptions/$(az account show --query id -o tsv)/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts?api-version=2024-10-01" --query "value[?kind==\'AIProject\'].name" -o json',
    shell=True, capture_output=True, text=True
).stdout.strip()

if _projects_json:
    _project_names = json.loads(_projects_json)
    if _project_names:
        _project_name = _project_names[0]
    else:
        _project_name = None
else:
    _project_name = None

if _project_name:
    PROJECT_ENDPOINT = f"{AI_SERVICES_ENDPOINT.rstrip('/')}/api/projects/{_project_name}"
    print(f"✓ Project Name: {_project_name}")
else:
    # Fallback: construct from AI Services name pattern
    _suffix = AI_SERVICES_NAME.replace("aiservices", "")
    _project_name = f"project{_suffix}"
    PROJECT_ENDPOINT = f"{AI_SERVICES_ENDPOINT.rstrip('/')}/api/projects/{_project_name}"
    print(f"⚠ Could not auto-discover project. Using fallback: {_project_name}")

print(f"✓ PROJECT_ENDPOINT: {PROJECT_ENDPOINT}")

# --- MCP Container App FQDN ---
MCP_FQDN = az_query(
    f'az containerapp show -g {RESOURCE_GROUP} -n mcp-http-server --query "properties.configuration.ingress.fqdn" -o tsv',
    "MCP Container App FQDN (has deploy-mcp.sh been run?)"
)
MCP_SERVER_URL = f"https://{MCP_FQDN}/mcp"
print(f"✓ MCP Server URL: {MCP_SERVER_URL}")

# --- AI Search connection name ---
AI_SEARCH_NAME = az_query(
    f'az search service list -g {RESOURCE_GROUP} --query "[0].name" -o tsv',
    "AI Search service name"
)
# The connection name in the project is typically the AI Services name + 'search'
AI_SEARCH_CONNECTION_NAME = f"{AI_SERVICES_NAME}search"
print(f"✓ AI Search Service: {AI_SEARCH_NAME}")
print(f"✓ AI Search Connection Name: {AI_SEARCH_CONNECTION_NAME}")
print(f"✓ AI Search Index: {AI_SEARCH_INDEX_NAME}")

# --- VNet name (for reference) ---
VNET_NAME = az_query(
    f'az network vnet list -g {RESOURCE_GROUP} --query "[0].name" -o tsv',
    "VNet name"
)
print(f"✓ VNet: {VNET_NAME}")

# --- Public network access status ---
PUBLIC_ACCESS_STATUS = az_query(
    f'az cognitiveservices account show -g {RESOURCE_GROUP} -n {AI_SERVICES_NAME} --query "properties.publicNetworkAccess" -o tsv',
    "AI Services publicNetworkAccess status"
)
print(f"✓ Current Public Network Access: {PUBLIC_ACCESS_STATUS}")

# --- Summary ---
print("\n" + "=" * 60)
print("RESOURCE DISCOVERY COMPLETE")
print("=" * 60)
print(f"  AI Services:       {AI_SERVICES_NAME}")
print(f"  Project Endpoint:  {PROJECT_ENDPOINT}")
print(f"  MCP Server URL:    {MCP_SERVER_URL}")
print(f"  AI Search:         {AI_SEARCH_NAME} (connection: {AI_SEARCH_CONNECTION_NAME})")
print(f"  VNet:              {VNET_NAME}")
print(f"  Public Access:     {PUBLIC_ACCESS_STATUS}")
print(f"  Mode:              {CONNECTIVITY_MODE}")
print("=" * 60)

---
## Cell 3 — Switch to Public Access (public mode only)

**Skip this cell if using `vnet` mode.**

Temporarily enables public network access on the AI Services account so Cloud Shell can reach the Foundry API. Backend resources (AI Search, Cosmos DB, Storage, MCP server) remain fully private — the Data Proxy still routes agent tool calls through the VNet.

> **Important:** Remember to run the **Revert to Private Access** cell at the bottom when you're done testing.

In [ ]:
if CONNECTIVITY_MODE == "public":
    print("Enabling public network access on AI Services...")
    print(f"  Target: {AI_SERVICES_NAME}")

    result = subprocess.run(
        f'az cognitiveservices account update -g {RESOURCE_GROUP} -n {AI_SERVICES_NAME} '
        f'--custom-domain {AI_SERVICES_NAME} '
        f'--api-properties publicNetworkAccess=Enabled',
        shell=True, capture_output=True, text=True
    )

    if result.returncode != 0:
        # Try alternative approach via az rest
        sub_id = az_query('az account show --query id -o tsv', 'Subscription ID')
        resource_id = f"/subscriptions/{sub_id}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{AI_SERVICES_NAME}"

        result2 = subprocess.run(
            f'az rest --method patch --url "https://management.azure.com{resource_id}?api-version=2024-10-01" '
            f'--body \'{{"properties": {{"publicNetworkAccess": "Enabled", "networkAcls": {{"defaultAction": "Allow"}}}}}}\' ',
            shell=True, capture_output=True, text=True
        )
        if result2.returncode != 0:
            print(f"\n✗ Failed to enable public access.")
            print(f"  Stderr: {result2.stderr.strip()[:500]}")
            raise SystemExit("Cannot enable public access. Check permissions (Owner/Contributor required).")

    # Verify the change
    new_status = az_query(
        f'az cognitiveservices account show -g {RESOURCE_GROUP} -n {AI_SERVICES_NAME} --query "properties.publicNetworkAccess" -o tsv',
        "Verify public access status after update"
    )
    if new_status.lower() == "enabled":
        print(f"\n✓ Public network access is now ENABLED on {AI_SERVICES_NAME}")
        print("  Remember to run the 'Revert to Private Access' cell when done.")
    else:
        print(f"\n⚠ Public access status is '{new_status}' — expected 'Enabled'. Check manually.")
else:
    print("VNet mode — skipping public access toggle.")
    print("Make sure Cloud Shell has VNet integration configured.")

---
## Cell 4 — Install Dependencies

In [ ]:
%pip install azure-ai-projects azure-identity openai --quiet

---
## Cell 5 — Initialize SDK Clients & Helpers

Creates the shared `credential`, `project_client`, and `openai_client`. Also defines helper functions used by all tests.

In [ ]:
import logging
import time

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logging.getLogger("azure.core.pipeline.policies.http_logging_policy").setLevel(logging.INFO)
logging.getLogger("httpx").setLevel(logging.INFO)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MCPTool,
    PromptAgentDefinition,
    AzureAISearchAgentTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)
from azure.identity import DefaultAzureCredential
from openai.types.responses import ResponseInputParam
from openai.types.responses.response_input_param import McpApprovalResponse

# --- Initialize clients ---
credential = DefaultAzureCredential()
project_client = AIProjectClient(credential=credential, endpoint=PROJECT_ENDPOINT)
openai_client = project_client.get_openai_client()

print(f"✓ AIProjectClient connected to: {PROJECT_ENDPOINT}")
print(f"✓ OpenAI client ready (model: {MODEL_NAME})")

# --- Test results tracker ---
test_results = {}


def log_response_info(response, label="Response"):
    """Extract and log useful debugging info from OpenAI response objects."""
    try:
        if hasattr(response, '_request_id'):
            print(f"  {label} - Request ID: {response._request_id}")
        if hasattr(response, 'id'):
            print(f"  {label} - Response ID: {response.id}")
    except Exception:
        pass


def log_exception_info(exception, label="Exception"):
    """Extract and log request info from exceptions for debugging."""
    try:
        if hasattr(exception, 'response') and exception.response is not None:
            resp = exception.response
            headers = resp.headers if hasattr(resp, 'headers') else {}
            print(f"  {label} - x-request-id: {headers.get('x-request-id', 'N/A')}")
            print(f"  {label} - x-ms-request-id: {headers.get('x-ms-request-id', 'N/A')}")
            if hasattr(resp, 'status_code'):
                print(f"  {label} - HTTP Status: {resp.status_code}")
        if hasattr(exception, 'request_id'):
            print(f"  {label} - request_id: {exception.request_id}")
    except Exception:
        pass


def cleanup_agent(agent):
    """Safely delete an agent version."""
    if agent is not None:
        try:
            project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
            print(f"  Cleaned up agent: {agent.name} v{agent.version}")
        except Exception as e:
            print(f"  ⚠ Agent cleanup failed: {e}")


print("\n✓ Helpers and SDK clients initialized")

---
## Test 1 — OpenAI Responses API (Direct)

Validates basic connectivity to the AI Services endpoint by calling the Responses API directly (no agent, no tools). This must pass before running agent tests.

In [ ]:
print("=" * 60)
print("TEST 1: OpenAI Responses API (Direct)")
print("=" * 60)

try:
    response = openai_client.responses.create(
        model=MODEL_NAME,
        input="What is 2 + 2? Answer with just the number.",
    )
    log_response_info(response, "Direct API")

    print(f"\n✓ Response: {response.output_text}")
    print("\n✓ TEST 1 PASSED: OpenAI Responses API works")
    test_results['1_responses_api'] = True

except Exception as e:
    print(f"\n✗ TEST 1 FAILED: {e}")
    log_exception_info(e, "API Error")
    test_results['1_responses_api'] = False
    import traceback; traceback.print_exc()

---
## Test 2 — Basic Agent (No Tools)

Creates an agent using `PromptAgentDefinition`, sends a prompt via the Responses API with conversation threads, and validates the response. No external tools — just validates the Agents v2 SDK pattern works.

In [ ]:
print("=" * 60)
print("TEST 2: Basic Agent (No Tools)")
print("=" * 60)

agent = None
try:
    agent = project_client.agents.create_version(
        agent_name="basic-test-agent",
        definition=PromptAgentDefinition(
            model=MODEL_NAME,
            instructions="You are a helpful assistant. Answer briefly and concisely.",
        ),
    )
    print(f"✓ Created agent (id: {agent.id}, name: {agent.name}, version: {agent.version})")

    conversation = openai_client.conversations.create()
    print(f"✓ Created conversation: {conversation.id}")

    response = openai_client.responses.create(
        conversation=conversation.id,
        input="Say hello and confirm you are working. Keep it brief.",
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    )
    log_response_info(response, "Basic Agent")

    print(f"\n✓ Agent response: {response.output_text}")
    print("\n✓ TEST 2 PASSED: Basic agent works with Responses API")
    test_results['2_basic_agent'] = True

except Exception as e:
    print(f"\n✗ TEST 2 FAILED: {e}")
    log_exception_info(e, "Basic Agent Error")
    test_results['2_basic_agent'] = False
    import traceback; traceback.print_exc()

finally:
    cleanup_agent(agent)

---
## Test 3 — MCP Tool via Agent

Creates an agent with an `MCPTool` pointing to the private weather MCP server. The agent sends a weather query which routes through:

```
Cloud Shell → AI Services API → Data Proxy → Private VNet → mcp-subnet → weather-mcp Container App
```

**Known issue:** ~50% of requests may fail with `TaskCanceledException` due to Hyena cluster routing (Data Proxy on only 1 of 2 scale units). The cell retries automatically up to `MAX_RETRIES` times.

In [ ]:
print("=" * 60)
print("TEST 3: MCP Tool via Agent")
print("=" * 60)
print(f"  MCP Server URL: {MCP_SERVER_URL}")
print(f"  Max retries: {MAX_RETRIES}")

test_results['3_mcp_tool'] = False

for attempt in range(1, MAX_RETRIES + 1):
    agent = None
    try:
        if attempt > 1:
            print(f"\n--- Retry {attempt}/{MAX_RETRIES} ---")
            time.sleep(2)

        mcp_tool = MCPTool(
            server_label="weather-mcp",
            server_url=MCP_SERVER_URL,
            require_approval="never",
        )

        agent = project_client.agents.create_version(
            agent_name="mcp-tool-test",
            definition=PromptAgentDefinition(
                model=MODEL_NAME,
                instructions="You are a helpful agent. When asked about weather, use the get_weather tool from the MCP server.",
                tools=[mcp_tool],
            ),
        )
        print(f"✓ Created agent with MCP tool (id: {agent.id})")

        conversation = openai_client.conversations.create()
        print(f"✓ Created conversation: {conversation.id}")

        print("  Sending request to use MCP get_weather tool...")
        response = openai_client.responses.create(
            conversation=conversation.id,
            input="What is the current weather in London? Use the get_weather tool.",
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        )
        log_response_info(response, "MCP Tool")

        # Handle MCP approval if needed
        for item in response.output:
            if hasattr(item, 'type') and item.type == "mcp_approval_request":
                print(f"  MCP approval requested for: {item.server_label}")
                input_list: ResponseInputParam = [
                    McpApprovalResponse(
                        type="mcp_approval_response",
                        approve=True,
                        approval_request_id=item.id,
                    )
                ]
                response = openai_client.responses.create(
                    input=input_list,
                    previous_response_id=response.id,
                    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
                )

        print(f"\n✓ Agent response: {response.output_text}")
        print(f"\n✓ TEST 3 PASSED: MCP tool via agent (attempt {attempt}/{MAX_RETRIES})")
        test_results['3_mcp_tool'] = True
        cleanup_agent(agent)
        break  # Success — no more retries

    except Exception as e:
        error_str = str(e)
        print(f"\n✗ Attempt {attempt}/{MAX_RETRIES} failed: {error_str[:200]}")
        log_exception_info(e, "MCP Tool Error")

        if "TaskCanceledException" in error_str:
            print("  ⚠ Known Hyena issue — Data Proxy hit wrong scale unit. Retrying...")
        elif "424" in error_str or "Failed Dependency" in error_str:
            print("  ⚠ DNS resolution issue — Data Proxy can't resolve private Container Apps DNS.")

        cleanup_agent(agent)

        if attempt == MAX_RETRIES:
            print(f"\n✗ TEST 3 FAILED after {MAX_RETRIES} attempts")
            import traceback; traceback.print_exc()

---
## Test 4 — AI Search Tool via Agent

Creates an agent with `AzureAISearchAgentTool` using SIMPLE query type, and sends a search query. The agent routes through the Data Proxy to the private AI Search endpoint.

> Requires a `test-index` with at least one document (see Step 3 of TESTING-GUIDE.md).

In [ ]:
print("=" * 60)
print("TEST 4: AI Search Tool via Agent")
print("=" * 60)
print(f"  Connection: {AI_SEARCH_CONNECTION_NAME}")
print(f"  Index: {AI_SEARCH_INDEX_NAME}")

agent = None
try:
    search_tool = AzureAISearchAgentTool(
        azure_ai_search=AzureAISearchToolResource(indexes=[
            AISearchIndexResource(
                project_connection_id=AI_SEARCH_CONNECTION_NAME,
                index_name=AI_SEARCH_INDEX_NAME,
                query_type=AzureAISearchQueryType.SIMPLE,
            )
        ])
    )

    agent = project_client.agents.create_version(
        agent_name="search-test-agent",
        definition=PromptAgentDefinition(
            model=MODEL_NAME,
            instructions="You are a helpful assistant. Use the search tool to find information.",
            tools=[search_tool],
        ),
    )
    print(f"✓ Created agent with AI Search tool (id: {agent.id})")

    conversation = openai_client.conversations.create()
    print(f"✓ Created conversation: {conversation.id}")

    response = openai_client.responses.create(
        conversation=conversation.id,
        input="Search for any documents in the index and tell me what you find.",
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    )
    log_response_info(response, "AI Search")

    print(f"\n✓ Agent response: {response.output_text[:500]}")
    print("\n✓ TEST 4 PASSED: AI Search tool queried private search endpoint")
    test_results['4_ai_search'] = True

except Exception as e:
    print(f"\n✗ TEST 4 FAILED: {e}")
    log_exception_info(e, "AI Search Error")
    test_results['4_ai_search'] = False
    import traceback; traceback.print_exc()

finally:
    cleanup_agent(agent)

---
## Test 5 — Direct MCP Connectivity (VNet mode only)

Tests raw HTTP connectivity to the private MCP server using the JSON-RPC session flow:
1. `initialize` → capture `mcp-session-id` header
2. `tools/list` → enumerate available tools
3. `tools/call` → execute `get_weather(city: "London")`

**This test only works when Cloud Shell has direct VNet access** (VNet integration or Bastion). In `public` mode the MCP server FQDN is not resolvable from the internet.

In [ ]:
print("=" * 60)
print("TEST 5: Direct MCP Connectivity (VNet only)")
print("=" * 60)

if CONNECTIVITY_MODE != "vnet":
    print("\n⏭ SKIPPED: Direct MCP connectivity requires VNet mode.")
    print("  The private MCP server FQDN is not resolvable from the internet.")
    print("  Agent tests (Test 3) still work because the Data Proxy routes")
    print("  through the VNet internally.")
    test_results['5_mcp_direct'] = None  # Skipped
else:
    import urllib.request
    import ssl

    try:
        ctx = ssl.create_default_context()
        print(f"  Target: {MCP_SERVER_URL}")

        # Step 1: Initialize
        print("\n--- Step 1: Initialize (get mcp-session-id) ---")
        init_data = json.dumps({
            "method": "initialize",
            "params": {
                "protocolVersion": "2025-11-25",
                "capabilities": {"sampling": {}, "elicitation": {}, "roots": {"listChanged": True}},
                "clientInfo": {"name": "test-mcp-client", "version": "1.0.0"}
            },
            "jsonrpc": "2.0", "id": 0
        }).encode('utf-8')

        init_req = urllib.request.Request(
            MCP_SERVER_URL, data=init_data,
            headers={"Content-Type": "application/json", "Accept": "application/json, text/event-stream"},
            method="POST"
        )
        with urllib.request.urlopen(init_req, timeout=15, context=ctx) as resp:
            body = resp.read().decode('utf-8')
            mcp_session_id = resp.getheader('mcp-session-id')
            print(f"  ✓ HTTP {resp.getcode()} — Response: {body[:200]}")
            if not mcp_session_id:
                raise RuntimeError("No mcp-session-id header returned")
            print(f"  ✓ Session ID: {mcp_session_id}")

        # Step 2: List Tools
        print("\n--- Step 2: List Tools ---")
        list_data = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}).encode('utf-8')
        list_req = urllib.request.Request(
            MCP_SERVER_URL, data=list_data,
            headers={"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "mcp-session-id": mcp_session_id},
            method="POST"
        )
        with urllib.request.urlopen(list_req, timeout=10, context=ctx) as resp:
            result = json.loads(resp.read().decode('utf-8'))
            tools = result.get("result", {}).get("tools", [])
            print(f"  ✓ Found {len(tools)} tool(s):")
            for t in tools:
                print(f"      - {t.get('name')}: {t.get('description', '')[:60]}")

        # Step 3: Call get_weather
        print("\n--- Step 3: Call get_weather('London') ---")
        call_data = json.dumps({
            "jsonrpc": "2.0", "id": 2,
            "method": "tools/call",
            "params": {"name": "get_weather", "arguments": {"city": "London"}}
        }).encode('utf-8')
        call_req = urllib.request.Request(
            MCP_SERVER_URL, data=call_data,
            headers={"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "mcp-session-id": mcp_session_id},
            method="POST"
        )
        with urllib.request.urlopen(call_req, timeout=10, context=ctx) as resp:
            body = resp.read().decode('utf-8')
            print(f"  ✓ HTTP {resp.getcode()} — Response: {body}")

        print("\n✓ TEST 5 PASSED: Direct MCP session flow working")
        test_results['5_mcp_direct'] = True

    except Exception as e:
        print(f"\n✗ TEST 5 FAILED: {e}")
        test_results['5_mcp_direct'] = False
        import traceback; traceback.print_exc()

---
## Test Summary

Prints pass/fail results for all tests.

In [ ]:
print("\n" + "=" * 60)
print("TEST SUMMARY")
print("=" * 60)

for name, passed in test_results.items():
    if passed is None:
        status = "⏭ SKIPPED"
    elif passed:
        status = "✓ PASSED"
    else:
        status = "✗ FAILED"
    print(f"  {name}: {status}")

# Count only non-skipped tests
run_results = {k: v for k, v in test_results.items() if v is not None}
passed_count = sum(1 for v in run_results.values() if v)
total_count = len(run_results)

print(f"\n  {passed_count}/{total_count} tests passed")

print("\n" + "=" * 60)
if all(v for v in run_results.values()):
    print("ALL TESTS PASSED")
else:
    print("SOME TESTS FAILED")
    if not test_results.get('3_mcp_tool'):
        print("  Tip: MCP test may fail ~50% due to Hyena routing. Re-run the MCP cell.")
print("=" * 60)

---
## Cleanup — Revert to Private Access

**Run this cell when you're done testing** to restore private network access on the AI Services account.

Only needed if you used `public` connectivity mode. In `vnet` mode this cell is a no-op.

In [ ]:
if CONNECTIVITY_MODE == "public":
    print("Reverting AI Services to private access...")
    print(f"  Target: {AI_SERVICES_NAME}")

    sub_id = az_query('az account show --query id -o tsv', 'Subscription ID')
    resource_id = f"/subscriptions/{sub_id}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{AI_SERVICES_NAME}"

    result = subprocess.run(
        f'az rest --method patch --url "https://management.azure.com{resource_id}?api-version=2024-10-01" '
        f'--body \'{{"properties": {{"publicNetworkAccess": "Disabled", "networkAcls": {{"defaultAction": "Deny"}}}}}}\' ',
        shell=True, capture_output=True, text=True
    )

    if result.returncode != 0:
        print(f"\n✗ Failed to revert public access.")
        print(f"  Stderr: {result.stderr.strip()[:500]}")
        print("  You can revert manually:")
        print(f"  az cognitiveservices account update -g {RESOURCE_GROUP} -n {AI_SERVICES_NAME} --api-properties publicNetworkAccess=Disabled")
    else:
        # Verify
        new_status = az_query(
            f'az cognitiveservices account show -g {RESOURCE_GROUP} -n {AI_SERVICES_NAME} --query "properties.publicNetworkAccess" -o tsv',
            "Verify public access status after revert"
        )
        print(f"\n✓ Public network access is now: {new_status}")
        if new_status.lower() == "disabled":
            print("  AI Services is private again.")
        else:
            print(f"  ⚠ Expected 'Disabled' but got '{new_status}'. Check manually.")
else:
    print("VNet mode — no public access to revert.")

# Close SDK clients
try:
    openai_client.close()
    project_client.close()
    credential.close()
    print("✓ SDK clients closed.")
except Exception:
    pass

print("\n✓ Cleanup complete.")